# 실험3-act — act+TE **K별** 학습+eval (bimamba 계열과 공정 비교용, 해준 담당)

기존 act+TE 는 K=100 뿐이라 K=20 등에서 bimamba 계열과 **공정 비교가 안 됨**.
→ 여기서 **act 도 K별로 학습+eval** 해서 `act+TE / bimamba+TE / bimamba+carry+TE` 를
**같은 K·같은 100ep·같은 TE(0.01)·같은 seed** 로 한 표에 놓는다.

- act 는 carry 없음 → 변형은 `act+TE` 하나. K=100 은 기존 `act` 폴더 재사용(학습 스킵).
- bimamba 결과(`bk{K}_*`)는 **`exp3_carry_te_ksweep`** 이 만든 것 — 그거 먼저(또는 같이) 돌려야 3자 표가 참.
- ★서버 나눠 쓰면 `K_LIST`·`SEEDS` 를 bimamba 쪽과 **똑같이** 맞출 것(안 그러면 비교 깨짐).


## 0) 부팅 + K별 act 태그 등록


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
v23 = cf.v23

# ── 무엇을 하나 ────────────────────────────────────────────────────────────
# act+TE 를 **K별로 학습+eval** → bimamba+TE / bimamba+carry+TE 와 **같은 K·같은 100ep**로 공정 비교.
#   (기존 act+TE 는 K=100 뿐이라 K=20 등에서 공정 비교 불가 → 여기서 act 도 K별로 맞춤)
# act 는 carry 없음(stateless) → 변형은 act+TE 하나. bimamba 결과는 exp3_carry_te_ksweep 이 만든 것.

TASK   = 'libero_10'
K_LIST = [10, 15, 20, 50, 100, 150]   # bimamba 쪽과 동일하게. ★서버별로 자기 몫만 편집.
SEEDS  = [0]                          # bimamba 실험과 같은 seed 로 맞추기!
N_EP   = 100                          # task당 100 → LIBERO-10 = 총 1000ep (bimamba 쪽과 동일)
GPUS   = v23.available_gpus()

# K=100 은 기존 'act' 폴더 재사용 → 학습 스킵. 나머지 K 는 새 act 태그 등록(carry 없음).
def src_tag(K):
    return 'act' if K == 100 else f'act_k{K}'
for K in K_LIST:
    t = src_tag(K)
    v23.MODEL_CONFIGS.setdefault(t, ('act', v23.LR, K, [], False))   # 순수 ACT, chunk_size=K
    v23.MODEL_DIR_NAMES.setdefault(t, t)

_TE = ['--policy.temporal_ensemble_coeff=0.01', '--policy.n_action_steps=1']
def out_tag(K):
    return f'ak{K}_te'                # act+TE eval 출력 태그
for K in K_LIST:
    v23.MODEL_DIR_NAMES.setdefault(out_tag(K), out_tag(K))

print('act K sweep:', K_LIST, '| seeds', SEEDS, '| N_EP', N_EP, '| GPU', GPUS)
print('학습 필요(K!=100):', [src_tag(K) for K in K_LIST if K != 100])

## 1) 학습 (K별 act — K=100·이미 된 것 스킵)


In [ ]:
# ── K별 act 학습 (K=100 은 기존 'act' 재사용→스킵, 이미 된 것도 스킵/이어서) ──
#   ★서버 나눠서 하면 K_LIST 를 자기 몫으로.
train_jobs = [(src_tag(K), s, TASK) for K in K_LIST if K != 100 for s in SEEDS]
if train_jobs:
    cf.run_training_jobs(train_jobs, GPUS, prefetch_task=TASK)
else:
    print('학습할 것 없음 (전부 K=100 재사용 or 이미 완료)')

## 2) eval (act+TE)


In [ ]:
# ── 각 K act 체크포인트에서 act+TE eval (TE 0.01, n_act=1). action 기록→떨림. 유효 완료 skip ──
_MINEP = 10 * N_EP // 2
def _has_valid(ot, s):
    info = v23.eval_clean_dir(ot, s, TASK) / 'eval_info.json'
    if not info.exists(): return False
    try:
        ov = json.loads(info.read_text()).get('overall', {})
        return (ov.get('n_ep', ov.get('n_episodes')) or 0) >= _MINEP
    except Exception:
        return False

jobs = [(K, s) for K in K_LIST for s in SEEDS if not _has_valid(out_tag(K), s)]
print(f'eval 실행 {len(jobs)} / 전체 {len(K_LIST)*len(SEEDS)} (유효 완료 skip)')
ng = len(GPUS)
for i in range(0, len(jobs), ng):
    chunk = jobs[i:i + ng]
    labeled = []
    for g, (K, s) in zip(GPUS, chunk):
        try:
            cmd = v23.make_eval_cmd(src_tag(K), seed=s, task=TASK, gpu_id=g, n_episodes=N_EP,
                                    select=cf.CKPT_STEP, extra_policy=_TE,
                                    out_dir=v23.eval_clean_dir(out_tag(K), s, TASK))
            labeled.append((f'{out_tag(K)}/seed{s}', cmd))
        except FileNotFoundError as e:
            print('  skip:', e)
    if labeled:
        print(f'\n===== eval 청크 {i//ng+1} ({len(labeled)} run) =====')
        v23.launch_cmds_live(labeled)
print('\n완료')

## 3) 공정 비교표 — K별 act+TE vs bimamba+TE vs bimamba+carry+TE


In [ ]:
# ── 공정 비교표: K별 act+TE vs bimamba+TE vs bimamba+carry+TE (같은 K·같은 100ep) ──
#   bimamba 결과(bk{K}_*)는 exp3_carry_te_ksweep 이 만든 것. 없으면 그 칸은 빈칸.
import numpy as np, smooth_metrics_paper as smp
importlib.reload(smp)
FS, STRIDE = cf.fps_of(TASK), 100
def rec(tag, s):
    d = cf.OUTPUT_BASE / 'eval_clean' / TASK / v23.MODEL_DIR_NAMES.get(tag, tag) / f'seed{s}'
    if not d.is_dir(): return None
    best = None
    for info in d.rglob('eval_info.json'):
        try: ov = json.loads(info.read_text()).get('overall', {})
        except Exception: continue
        n = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or n > best['n']:
            best = {'sr': ov.get('pc_success'), 'n': n, 'act': (info.parent/'actions').is_dir(), 'p': info.parent}
    return best
def sr_of(tag):
    vals = [rec(tag, s)['sr'] for s in SEEDS if rec(tag, s) and (rec(tag, s)['n'] or 0) >= 10*N_EP//2 and rec(tag, s)['sr'] is not None]
    return float(np.mean(vals)) if vals else None
def sm_of(tag):
    trajs = []
    for s in SEEDS:
        e = rec(tag, s)
        if e and (e['n'] or 0) >= 10*N_EP//2 and e['act']:
            t = v23._load_action_trajs(e['p']/'actions') or []
            if len(t) >= 80: trajs += t
    return smp.aggregate_paper(trajs, boundary_stride=STRIDE, fs=FS) if trajs else None

# 3자 태그: act+TE = ak{K}_te / bimamba+TE = bk{K}_nocarry_te / bimamba+carry+TE = bk{K}_carry_te
COLS = [('act+TE', lambda K: f'ak{K}_te'),
        ('bimamba+TE', lambda K: f'bk{K}_nocarry_te'),
        ('bimamba+carry+TE', lambda K: f'bk{K}_carry_te')]
def _s(v): return f'{v:.1f}' if v is not None else '-'

print('== SR vs K (같은 100ep) ==')
print(f'{"K":>5}' + ''.join(f'{c[0]:>20}' for c in COLS))
for K in K_LIST:
    print(f'{K:>5}' + ''.join(f'{_s(sr_of(c[1](K))):>20}' for c in COLS))

print('\n== 떨림 vs K (jerk / B/I / sflip) ==')
print(f'{"K":>5}  {"variant":<18}{"jerk":>9}{"bnd":>9}{"int":>9}{"B/I":>7}{"sflip":>9}{"n":>6}')
for K in K_LIST:
    for name, fn in COLS:
        a = sm_of(fn(K))
        if a:
            print(f'{K:>5}  {name:<18}{a["jerk_rms_mean"]:>9.4f}{a["boundary_jerk_rms_mean"]:>9.4f}'
                  f'{a["interior_jerk_rms_mean"]:>9.4f}{a["boundary_interior_ratio_mean"]:>7.2f}'
                  f'{a["sign_flip_rate_mean"]:>9.4f}{a["n_traj"]:>6}')
        else:
            print(f'{K:>5}  {name:<18}' + ' '*40 + '(빈칸)')
print('\n※ 세 모델 전부 같은 K·같은 100ep·TE 0.01·같은 seed → 공정 비교. bimamba 칸 빈칸이면 exp3_carry_te_ksweep 먼저.')